# 🚀 Walkability Semantic Segmentation: Kaggle Training on Tesla T4

This notebook trains **BiSeNetV2** using Kaggle's free **Tesla T4 GPU (16 GB VRAM)**.

### ⚙️ Required Kaggle Settings Before Running:
1. **Settings** -> **Accelerator** -> Select **GPU T4 x2** (or GPU T4)
2. **Settings** -> Ensure **Internet** is toggled **ON** (needed for git clone & pip install)

## 1. Verify GPU Setup

In [ ]:
!nvidia-smi

import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print(f"Total GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("❌ GPU is NOT active. Please go to Settings -> Accelerator -> GPU T4 x2")

## 2. Clone Repository & Checkout Branch

In [ ]:
import os
%cd /kaggle/working

if not os.path.exists("/kaggle/working/Capstone-Project"):
    !git clone https://github.com/dev-june05/Capstone-Project.git

%cd /kaggle/working/Capstone-Project
!git checkout divyansh/model-training
!git pull origin divyansh/model-training

## 3. Install Required Dependencies

In [ ]:
!pip install albumentations

## 4. Dataset Setup (Cityscapes)

You have two easy ways to get Cityscapes into Kaggle:

* **Method A (Easiest)**: Upload the two zip files as a Kaggle Dataset (**+ Add Data** -> **Upload Dataset** -> name it `cityscapes`). Kaggle automatically mounts and unzips it in `/kaggle/input/`!
* **Method B (Direct Download / Unzip)**: If you have direct download links (e.g. from Google Drive via `gdown`), you can download and unzip them directly into `data/cityscapes/`.

In [ ]:
import os
from pathlib import Path

target_dir = Path("/kaggle/working/Capstone-Project/data/cityscapes")
target_dir.mkdir(parents=True, exist_ok=True)

# Check if dataset was added via Kaggle '+ Add Data'
found_input = False
for root, dirs, files in os.walk("/kaggle/input"):
    if "leftImg8bit" in dirs and "gtFine" in dirs:
        found_input = True
        print(f"✅ Found Cityscapes dataset at: {root}")
        # Create symbolic links to prevent duplicating disk space
        !ln -sfn "{root}/leftImg8bit" /kaggle/working/Capstone-Project/data/cityscapes/leftImg8bit
        !ln -sfn "{root}/gtFine" /kaggle/working/Capstone-Project/data/cityscapes/gtFine
        print("Symlinks created successfully!")
        break

if not found_input:
    print("⚠️ Cityscapes not found in /kaggle/input.")
    print("If you haven't uploaded it yet, click '+ Add Data' at the top right of Kaggle to upload the 2 zip files.")

## 5. Verify Dataset Loading

In [ ]:
from utils.dataset import CityscapesDataset

train_ds = CityscapesDataset("data/cityscapes", split="train")
val_ds = CityscapesDataset("data/cityscapes", split="val")

print(f"✅ Training samples detected: {len(train_ds)} (expected: 2975)")
print(f"✅ Validation samples detected: {len(val_ds)} (expected: 500)")

## 6. Launch Model Training (BiSeNetV2 on T4 GPU)

In [ ]:
!python scripts/train.py --config configs/train_kaggle.yaml

## 7. Evaluate Trained Model

In [ ]:
!python scripts/evaluate.py --config configs/train_kaggle.yaml --checkpoint /kaggle/working/checkpoints/best.pth --split val